# Electricity consumption forecasting with XGBoost

A cleaned version of the XGBoost experiment from my summer 2022 internship. It compares calendar features with a one-step-ahead model that also uses observed past values. The original CSV was missing from the archive; this repository includes a reconstruction from the public UCI source. This notebook has no saved model results because it has not been re-executed.

**Evaluation:** chronological 70% train / 15% validation / 15% test. Validation controls early stopping; test is scored once. Lagged test features use *observed* prior values, so this is one-step-ahead evaluation, not an unattended multi-month forecast.


In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error


In [ ]:
from pathlib import Path
import pandas as pd

candidate_paths = [Path("../data/clean_hourly_data.csv"), Path("data/clean_hourly_data.csv")]
data_path = next((path for path in candidate_paths if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("The reconstructed CSV is missing; see data/README.md")

data = pd.read_csv(data_path, parse_dates=["ds"])
required = {"ds", "y"}
if not required.issubset(data.columns):
    raise ValueError("The CSV must contain ds (timestamp) and y (numeric target) columns")
data = data[["ds", "y"]].copy().sort_values("ds").reset_index(drop=True)
data["y"] = pd.to_numeric(data["y"], errors="raise")
if data.isna().any().any() or data["ds"].duplicated().any():
    raise ValueError("Remove missing values and duplicate timestamps before running these notebooks")
if not data["ds"].diff().dropna().eq(pd.Timedelta(hours=1)).all():
    raise ValueError("These notebooks require consecutive hourly observations")

train_end = int(len(data) * 0.70)
validation_end = int(len(data) * 0.85)
if train_end < 48 or validation_end - train_end < 24 or len(data) - validation_end < 24:
    raise ValueError("The dataset is too short for the chronological split")
print(f"Rows: {len(data):,}; train: {train_end:,}; validation: {validation_end - train_end:,}; test: {len(data) - validation_end:,}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

def report(y_true, y_pred, timestamps, title):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    print(f"{title} | MAE: {mae:.3f} | RMSE: {rmse:.3f}")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(timestamps, y_true, label="Observed", linewidth=1)
    ax.plot(timestamps, y_pred, label="Predicted", linewidth=1)
    ax.set(title=title, xlabel="Time", ylabel="Hourly sum of minute-level kW")
    ax.legend()
    fig.tight_layout()
    plt.show()


## Calendar-only baseline

These features are known in advance, so this model can predict the full test horizon without observing test targets.


In [ ]:
def calendar_features(timestamps):
    return pd.DataFrame({
        "hour": timestamps.dt.hour,
        "day_of_week": timestamps.dt.dayofweek,
        "day_of_year": timestamps.dt.dayofyear,
        "month": timestamps.dt.month,
        "year": timestamps.dt.year,
    })

calendar = calendar_features(data["ds"])
target = data["y"]
calendar_model = xgb.XGBRegressor(
    objective="reg:squarederror", n_estimators=1000, max_depth=5,
    learning_rate=0.05, random_state=42, tree_method="hist",
    eval_metric="mae", early_stopping_rounds=30,
)
calendar_model.fit(
    calendar.iloc[:train_end], target.iloc[:train_end],
    eval_set=[(calendar.iloc[train_end:validation_end], target.iloc[train_end:validation_end])],
    verbose=False,
)
calendar_prediction = calendar_model.predict(calendar.iloc[validation_end:])
report(target.iloc[validation_end:], calendar_prediction,
       data["ds"].iloc[validation_end:], "XGBoost: calendar features")


## Lagged observations

The lag model uses the most recent measured values. A 24-hour lag refers to the value exactly 24 rows earlier, which the loader verifies is 24 hours earlier.


In [ ]:
lag_hours = [1, 2, 3, 24, 24 * 7]
lagged = calendar.copy()
for hours in lag_hours:
    lagged[f"lag_{hours}h"] = target.shift(hours)
valid = lagged.notna().all(axis=1)
train_rows = valid & (data.index < train_end)
validation_rows = valid & (data.index >= train_end) & (data.index < validation_end)
test_rows = valid & (data.index >= validation_end)

lag_model = xgb.XGBRegressor(
    objective="reg:squarederror", n_estimators=1000, max_depth=5,
    learning_rate=0.05, random_state=42, tree_method="hist",
    eval_metric="mae", early_stopping_rounds=30,
)
lag_model.fit(
    lagged.loc[train_rows], target.loc[train_rows],
    eval_set=[(lagged.loc[validation_rows], target.loc[validation_rows])],
    verbose=False,
)
lag_prediction = lag_model.predict(lagged.loc[test_rows])
print(f"Previous-hour baseline MAE: {mean_absolute_error(target.loc[test_rows], lagged.loc[test_rows, 'lag_1h']):.3f}")
report(target.loc[test_rows], lag_prediction, data.loc[test_rows, "ds"],
       "XGBoost: observed lags + calendar")


## Interpretation

Compare the lag model with the previous-hour baseline. Its test result must not be compared directly with the calendar-only full-horizon result: the lag model has access to observed values inside the test period.
